# Part 1


## Introduction

An online resource was consulted for the purposes of rendering gymnasium environments in jupyter notebooks. This resource can be found at the following link and is referenced multiple times throughout this assignment:

- Source A: https://community.latenode.com/t/how-to-render-gymnasium-environment-inside-jupyter-without-external-window/30582/4

In [1]:
# =============================== Taken from Source A ===================
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "rl_suite").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

from IPython import display
import matplotlib.pyplot as plt
%matplotlib inline
# =======================================================================
from abc import ABC, abstractmethod
import gymnasium as gym
import numpy as np
import pprint  # useful for printing nested items

from rl_suite import (
    discretize_interval,
    even_bin_count,
    print_discrete_space,
    run_cartpole_episode,
)

In [2]:
ENV_ID, SEED = "CartPole-v1", 10
NUM_TIMESTEPS_GOAL = 10000
env = gym.make(ENV_ID, max_episode_steps=NUM_TIMESTEPS_GOAL)


def execute_environment(select_action, behaviour=None):
    return run_cartpole_episode(env, select_action, behaviour)

In [3]:
# ``print_discrete_space`` is imported from ``rl_suite`` in the Setup cell.

## Approach

It is clear that the duration of this control algorithm depends directly on how we discretize the continuous feedback; so our execution loop will be a function of the fineness of discretization. 

The algorithm for off-policy MC control algorithm involves using an infinite amount of episodes but this is obviously infeasible. We need to generate enough episodes that the agent can actually solve/optimize the problem (the target policy must converge to a deterministic optimum). The instructions say the goal is for the solver to balance it for 10,000 steps. Then our goal is to keep generating episodes until we finally reach an episode that is 10,000 timesteps long, however this is impractical so we implemented a maximum number of iterations of 100,000. Since the behaviour policy used to generate the episodes is arbitrarily soft and independent of the target policy being learned (except for coverage), we cannot use these episodes to test whether our algorithm has converged. So for every 1000 iterations, we generate an episode using the target policy as a test. Therefore, the agent has 100 "test attempts" to balance the cart for 10,000 timesteps, and 100,000 episodes to learn from. If the agent still does not accomplish this, it has failed.

We implemented three modified versions of the Off-Policy Control algorithm; each agent failed. We will present each implementation, discuss the changes, and afterward discuss the limitations of this approach. The "logs" for these algorithms can be found in Appendices B, C, and D.

Here is the main loop used to execute the environment (generate an episode) for agents 1 and 2:

In [4]:
# ``execute_environment`` is ``run_cartpole_episode`` from ``rl_suite`` (see Setup).

Here is an abstract agent class that implements the policy but doesnt implement the discretization of the environment (these will be implemented by the child classes).

In [5]:
class AbstractAgent(ABC):

    def __init__(self, gamma):
        self.gamma = gamma
        self.MAX_ITERATIONS = 100000

    @abstractmethod
    def discretize_spaces(self):
        pass

    @abstractmethod
    def get_discrete(self):
        pass

    def select_action(self, state, b=None):
        s = self.get_discrete(state)
        if b is not None:
            return b(s)
        return self.pi[s]
    
    def greedy_update(self):
        self.pi = np.argmax(self.table[..., 0], axis=-1)

    def behavioral(self, state=None, action=None):
        if state is not None and action is not None:
            return 0.5
        return np.random.choice(2, 1)[0] # choose between action 0 and 1 with equal prob

    def control(self):

        self.table = np.random.random_sample(self.table_dims)
        self.table[..., 1] = 0
        self.greedy_update()

        output_logs = []
        success = False

        for num_iter in range(1, self.MAX_ITERATIONS+1):

            if not num_iter % 1000:
                ep = execute_environment(self.select_action)
                output_logs.append(f"Iteration {num_iter}, Test {num_iter // 1000}: target policy episode length {len(ep)}")
                if len(ep) == NUM_TIMESTEPS_GOAL:
                    success = True
                    break
            
            G,W = 0,1
            b = self.behavioral
            
            episode = execute_environment(self.select_action, b)
            for t in range(len(episode)-2,-1,-1):
                G = G*self.gamma + episode[t+1][-1]
                cont_state, A, _ = episode[t]
                S = self.get_discrete(cont_state)
                Q, C = self.table[S][A]
                C += W
                Q = Q + (W/C)*(G-Q)
                self.table[S][A] = [Q,C]
                
                optimal_action = np.argmax(self.table[S][...,0])
                self.pi[S] = optimal_action
                if optimal_action != A:
                    continue
                W *= 1/b(S,A)
        
        if success:
            print(f"Success! The agent was able to balance the pole for at least {NUM_TIMESTEPS_GOAL} timesteps.")
        else:
            print(f"Failure to meet goal after {self.MAX_ITERATIONS} iterations.")
        return output_logs

## Agent 1

Here is our first agent for Off-Policy MC agent implementation. It is a child of the Abstract class above and implements discretization. The observed feedback includes four values: cart position, cart velocity, angle (in radians), and angular velocity. Since it is infeasible to have an infinite state space for MC control, we manually limit the range of the cart velocity and the angular velocity. We chose the smallest range that included every single observed value from initial testing; these were $(-2.5,2.5)$ and $(-3.5,3.5)$ for cart velocity and angular velocity, respectively. 

As mentioned earlier, discretization is necessary, and we decided to settle on 10 "bins" for each variable (so each variable has 10 possible values). Thus, the size of our state space is $|S| = 10^4 = 10000$, and since there are only two possible actions (push left or push right), our state-action is of size $|S \times A| = 10000 \times 2 = 20000$. For each state variables we ensure that there is an equal number of bins representing negative values as positive values (the edge case of 0 is irrelevant). Thus the number of bins are forced to be even.

The behaviour policy simply chooses between action 0 and 1 with equal probability for any state. Thus it is a soft policy and the assumption of coverage is still valid (any state-action pair possible under the target policy is possible under behaviour).

In [6]:
class Agent1(AbstractAgent):

    def __init__(self, gamma=0.9, num_bins=10, vel_range=(-2.5,2.5), angular_vel_range=(-3.5,3.5)):
        super().__init__(gamma)
        self.discretize_spaces(num_bins, vel_range, angular_vel_range)

    def _fix_num_bins(self, num_bins):
        return even_bin_count(num_bins)

    def _discretize(self, interval):
        return discretize_interval(self.n, interval)

    def discretize_spaces(self, num_bins, vel_range, angular_vel_range):
        self.n = self._fix_num_bins(num_bins)
        self.table_dims = (self.n,self.n,self.n,self.n,2,2)
        pos_range = (-2.4, 2.4) # non terminal range for position
        angle_range = (-0.2095, 0.2095) # non terminal range for angle (radians)
        intervals = [pos_range, vel_range, angle_range, angular_vel_range]
        self.discrete_space = [self._discretize(x) for x in intervals]

    def get_discrete(self, s):
        def mapper(i):
            bin_index = np.digitize(s[i], self.discrete_space[i])-1
            return min(bin_index, self.n-1) # edge case for max val of interval
        return tuple(map(mapper, range(len(s))))

In [7]:
agent1 = Agent1()
print_discrete_space(agent1.discrete_space)
agent1_output = agent1.control()

Cart Position: [-2.4  -1.92 -1.44 -0.96 -0.48  0.    0.48  0.96  1.44  1.92  2.4 ]

Cart Velocity: [-2.5 -2.  -1.5 -1.  -0.5  0.   0.5  1.   1.5  2.   2.5]

Pole Angle: [-0.2095 -0.1676 -0.1257 -0.0838 -0.0419  0.      0.0419  0.0838  0.1257
  0.1676  0.2095]

Pole Angular Velocity: [-3.5 -2.8 -2.1 -1.4 -0.7  0.   0.7  1.4  2.1  2.8  3.5]



KeyboardInterrupt: 

## Agent 2

Agent 1 failed to converge to an optimal target policy. We realized that the range of starting values enforced by the environment is very small (from -0.05 to 0.05). This meant that for each of our variables, the starting value could only belong to one of the bins. While this is fine theoretically (we do not need the Exploring Starts assumption for Off-Policy MC), it menas that very few of the states are being sampled frequently to make learning practical (theoretically we need to be able to guarantee each state-action pair is visited infinitely). Given our current discretization this would take an infeasibly long time unless we significantly increase the number of bins (which quickly becomes computationally intractable without distributed architecture).

We then remembered that there are only two possible actions: to push left or right. There is no way for the agent to decide how hard to push at any given timestep; it can only apply a pre-determined constant amount of pressure regardless of any state variables. The only thing the agent needs to decide is the direction to move the cart to. Therefore, the cart velocity and angular velocity are practically useless information! The only pertinent information for deciding on a direction to push are the position of the cart and angle of the pole (regardless of velocity). 

We will now only consider cart position and pole angle and use $100$ bins for each. Therefore the size of our state space $|S| = 100 \times 100 = 10000$ remains the same. By eliminating the useless velocity variables, we have gained a drastically finer discretization without an increase in the size of the state space!

Here is Agent 2:

In [ ]:
class Agent2(AbstractAgent):

    def __init__(self, gamma=0.9, num_bins=100):
        super().__init__(gamma)
        self.discretize_spaces(num_bins)

    def _fix_num_bins(self, num_bins):
        return even_bin_count(num_bins)

    def _discretize(self, interval):
        return discretize_interval(self.n, interval)

    def discretize_spaces(self, num_bins):
        self.n = self._fix_num_bins(num_bins)
        self.table_dims = (self.n,self.n,2,2)
        pos_range = (-2.4, 2.4) # non terminal range for position
        angle_range = (-0.2095, 0.2095) # non terminal range for angle (radians)
        intervals = [pos_range, angle_range]
        self.discrete_space = [self._discretize(x) for x in intervals]


    def get_discrete(self, state):
        def mapper(i):
            bin_index = np.digitize(s[i], self.discrete_space[i])-1
            return min(bin_index, self.n-1) # edge case for max val of interval
        s = (state[0],state[2]) if len(state) > 2 else state
        return tuple(map(mapper, range(len(s))))



In [ ]:
agent2 = Agent2()
print_discrete_space(agent2.discrete_space)
agent2_output = agent2.control()

Cart Position: [-2.4   -2.352 -2.304 -2.256 -2.208 -2.16  -2.112 -2.064 -2.016 -1.968
 -1.92  -1.872 -1.824 -1.776 -1.728 -1.68  -1.632 -1.584 -1.536 -1.488
 -1.44  -1.392 -1.344 -1.296 -1.248 -1.2   -1.152 -1.104 -1.056 -1.008
 -0.96  -0.912 -0.864 -0.816 -0.768 -0.72  -0.672 -0.624 -0.576 -0.528
 -0.48  -0.432 -0.384 -0.336 -0.288 -0.24  -0.192 -0.144 -0.096 -0.048
  0.     0.048  0.096  0.144  0.192  0.24   0.288  0.336  0.384  0.432
  0.48   0.528  0.576  0.624  0.672  0.72   0.768  0.816  0.864  0.912
  0.96   1.008  1.056  1.104  1.152  1.2    1.248  1.296  1.344  1.392
  1.44   1.488  1.536  1.584  1.632  1.68   1.728  1.776  1.824  1.872
  1.92   1.968  2.016  2.064  2.112  2.16   2.208  2.256  2.304  2.352
  2.4  ]

Cart Velocity: [-0.2095  -0.20531 -0.20112 -0.19693 -0.19274 -0.18855 -0.18436 -0.18017
 -0.17598 -0.17179 -0.1676  -0.16341 -0.15922 -0.15503 -0.15084 -0.14665
 -0.14246 -0.13827 -0.13408 -0.12989 -0.1257  -0.12151 -0.11732 -0.11313
 -0.10894 -0.10475 -0.10056 -0.

## Agent 3

Agent 2 also failed to converge despite our significantly finer discretization. We came up with the hypothesis that there is too much noise in our behaviour policy; by choosing either action with equal probability this policy is ignoring new information about the state-action pairs which is used to update the target policy. So then our final idea is to bring the behaviour policy closer to the target policy whilst retaining its "softness". This is done by providing a small epsilon term to the algorithm. At any point in an episode, the behaviour policy will choose the current best action (greedy argmax from target policy) with probability $1-\epsilon$, and with probability $\epsilon$ it will pick the non-greedy action for exploration purposes. Thus our behaviour policy is now an $\epsilon$-soft policy, but it still maintains coverage of the target policy.

To do this, however, we need to keep track of the probability for each action chosen by the behaviour policy. This was not needed before because both actions had an equal probability so we could simply use a hard code value of $0.5$. Clearly, $\epsilon \neq 1 - \epsilon \neq 0.5$, so we make a slight modification to our environment execution code (note that we are not changing the environment, we are only changing the information the agent keeps track of when an episode is generated). We also decided to decrease our gamma value (thereby decreasing the long term return and ultimately punishing the agent for shorter episodes).

In [ ]:
# Same episode runner as above — unified implementation handles Agents 1–3 (see ``rl_suite.utils.cartpole``).

Though this agent is a child class of the previous agent, this change in the behaviour policy requires modifications in class methods which have already been implemented: 

In [ ]:
class Agent3(Agent2):

    def __init__(self, gamma=0.3, epsilon=0.1, num_bins=100):
        super().__init__(gamma, num_bins)
        self.epsilon = epsilon


    def behavioral(self, state=None, action=None):
        s = self.get_discrete(state)
        greedy = self.pi[s]
        non_greedy = greedy ^ 1
        weights = (1-self.epsilon, self.epsilon)
        action = np.random.choice((greedy, non_greedy), p=weights)
        return (action, weights[int(action == non_greedy)])


    def control(self):
        self.table = np.random.random_sample(self.table_dims)
        self.table[..., 1] = 0
        self.greedy_update()

        output_logs = []
        success = False
        for num_iter in range(1, self.MAX_ITERATIONS+1):

            if not num_iter % 1000:
                ep = execute_environment(self.select_action)
                output_logs.append(f"Iteration {num_iter}, Test {num_iter // 1000}: target policy episode length {len(ep)}")
                if len(ep) == NUM_TIMESTEPS_GOAL:
                    success = True
                    break
            
            G,W = 0,1
            b = self.behavioral
            episode = execute_environment(self.select_action, b)
            for t in range(len(episode)-2,-1,-1):
                G = G*self.gamma + episode[t+1][-1]
                cont_state, A, prob, _ = episode[t]
                S = self.get_discrete(cont_state)
                Q, C = self.table[S][A]
                C += W
                Q = Q + ((W/C)*(G-Q))
                self.table[S][A] = [Q,C]
                
                optimal_action = np.argmax(self.table[S][...,0])
                self.pi[S] = optimal_action
                if optimal_action != A:
                    continue
                W *= 1/prob
        
        if success:
            print(f"Success! The agent was able to balance the pole for at least {NUM_TIMESTEPS_GOAL} timesteps.")
        else:
            print(f"Failure to meet goal after {self.MAX_ITERATIONS} iterations.")
        return output_logs

In [ ]:
agent3 = Agent3()
print_discrete_space(agent3.discrete_space)  # same as Agent 2
agent3_output = agent3.control()

Cart Position: [-2.4   -2.352 -2.304 -2.256 -2.208 -2.16  -2.112 -2.064 -2.016 -1.968
 -1.92  -1.872 -1.824 -1.776 -1.728 -1.68  -1.632 -1.584 -1.536 -1.488
 -1.44  -1.392 -1.344 -1.296 -1.248 -1.2   -1.152 -1.104 -1.056 -1.008
 -0.96  -0.912 -0.864 -0.816 -0.768 -0.72  -0.672 -0.624 -0.576 -0.528
 -0.48  -0.432 -0.384 -0.336 -0.288 -0.24  -0.192 -0.144 -0.096 -0.048
  0.     0.048  0.096  0.144  0.192  0.24   0.288  0.336  0.384  0.432
  0.48   0.528  0.576  0.624  0.672  0.72   0.768  0.816  0.864  0.912
  0.96   1.008  1.056  1.104  1.152  1.2    1.248  1.296  1.344  1.392
  1.44   1.488  1.536  1.584  1.632  1.68   1.728  1.776  1.824  1.872
  1.92   1.968  2.016  2.064  2.112  2.16   2.208  2.256  2.304  2.352
  2.4  ]

Cart Velocity: [-0.2095  -0.20531 -0.20112 -0.19693 -0.19274 -0.18855 -0.18436 -0.18017
 -0.17598 -0.17179 -0.1676  -0.16341 -0.15922 -0.15503 -0.15084 -0.14665
 -0.14246 -0.13827 -0.13408 -0.12989 -0.1257  -0.12151 -0.11732 -0.11313
 -0.10894 -0.10475 -0.10056 -0.

## Limitations

So why did our algorithms fail to converge to a deterministic optimal policy? Perhaps an even finer discretization (larger state space) is needed. perhaps a more selective discretization that involves non-linear transformations of the current intervals are needed (to emphasize states that are less likely to be encountered).

It is also important to note that this is a very limited environment; as mentioned earlier, the only actions the agent can take is to decide the direction in which to push the cart. The agent cannot specify the amount of force to apply at any timestep, nor can it decide not to interfere. It seems nearly impossible that a constant force applied at each timestep could ever enable control of this problem. As the documentation notes, the "center of gravity of the pole varies the amount of energy needed to move the cart underneath it". Perhaps this problem with the given state and action space could be better solved by non-tabular RL methods. Our conclusion is that for any commercial PC and GPU, solving this given problem (keeping the pole balanced indefinitely) with this algorithm and state-action space, is not possible.

# Appendix A

Here are the logs from the Agent 1 target policy "tests".

In [ ]:
pprint.pprint(agent1_output)

['Iteration 1000, Test 1: target policy episode length 30',
 'Iteration 2000, Test 2: target policy episode length 13',
 'Iteration 3000, Test 3: target policy episode length 10',
 'Iteration 4000, Test 4: target policy episode length 11',
 'Iteration 5000, Test 5: target policy episode length 10',
 'Iteration 6000, Test 6: target policy episode length 24',
 'Iteration 7000, Test 7: target policy episode length 15',
 'Iteration 8000, Test 8: target policy episode length 12',
 'Iteration 9000, Test 9: target policy episode length 10',
 'Iteration 10000, Test 10: target policy episode length 10',
 'Iteration 11000, Test 11: target policy episode length 12',
 'Iteration 12000, Test 12: target policy episode length 21',
 'Iteration 13000, Test 13: target policy episode length 12',
 'Iteration 14000, Test 14: target policy episode length 9',
 'Iteration 15000, Test 15: target policy episode length 10',
 'Iteration 16000, Test 16: target policy episode length 20',
 'Iteration 17000, Test 17:

# Appendix B

Here are the logs from the Agent 2 target policy "tests":

In [ ]:
pprint.pprint(agent2_output)

['Iteration 1000, Test 1: target policy episode length 23',
 'Iteration 2000, Test 2: target policy episode length 62',
 'Iteration 3000, Test 3: target policy episode length 29',
 'Iteration 4000, Test 4: target policy episode length 11',
 'Iteration 5000, Test 5: target policy episode length 30',
 'Iteration 6000, Test 6: target policy episode length 33',
 'Iteration 7000, Test 7: target policy episode length 13',
 'Iteration 8000, Test 8: target policy episode length 15',
 'Iteration 9000, Test 9: target policy episode length 18',
 'Iteration 10000, Test 10: target policy episode length 71',
 'Iteration 11000, Test 11: target policy episode length 23',
 'Iteration 12000, Test 12: target policy episode length 44',
 'Iteration 13000, Test 13: target policy episode length 23',
 'Iteration 14000, Test 14: target policy episode length 13',
 'Iteration 15000, Test 15: target policy episode length 19',
 'Iteration 16000, Test 16: target policy episode length 74',
 'Iteration 17000, Test 17

# Appendix C

Here are the logs from the Agent 3 target policy "tests":

In [ ]:
pprint.pprint(agent3_output)

['Iteration 1000, Test 1: target policy episode length 13',
 'Iteration 2000, Test 2: target policy episode length 11',
 'Iteration 3000, Test 3: target policy episode length 18',
 'Iteration 4000, Test 4: target policy episode length 20',
 'Iteration 5000, Test 5: target policy episode length 17',
 'Iteration 6000, Test 6: target policy episode length 10',
 'Iteration 7000, Test 7: target policy episode length 20',
 'Iteration 8000, Test 8: target policy episode length 17',
 'Iteration 9000, Test 9: target policy episode length 31',
 'Iteration 10000, Test 10: target policy episode length 12',
 'Iteration 11000, Test 11: target policy episode length 10',
 'Iteration 12000, Test 12: target policy episode length 15',
 'Iteration 13000, Test 13: target policy episode length 15',
 'Iteration 14000, Test 14: target policy episode length 12',
 'Iteration 15000, Test 15: target policy episode length 26',
 'Iteration 16000, Test 16: target policy episode length 12',
 'Iteration 17000, Test 17